In [1]:
import pandas as pd
import numpy as np
import tidyfinance as tf

In [6]:
prices = tf.download_data(
  domain="stock_prices", 
  symbols="AAPL",
  start_date="2000-01-01", 
  end_date="2023-12-31"
)
prices.head().round(3)

Failed to retrieve data for AAPL (Status code: 429)


""


In [3]:
from plotnine import *

In [4]:
apple_prices_figure = (
  ggplot(prices, aes(y="adjusted_close", x="date"))
  + geom_line()
  + labs(x="", y="", title="Apple stock prices from 2000 to 2023")
)
apple_prices_figure.show()

PlotnineError: "Could not evaluate the 'x' mapping: 'date' (original error: name 'date' is not defined)"

In [ ]:
returns = (prices
  .sort_values("date")
  .assign(ret=lambda x: x["adjusted_close"].pct_change())
  .get(["symbol", "date", "ret"])
)
returns

In [ ]:
returns = returns.dropna() 

In [ ]:
from mizani.formatters import percent_format

quantile_05 = returns["ret"].quantile(0.05)

apple_returns_figure = (
  ggplot(returns, aes(x="ret"))
  + geom_histogram(bins=100)
  + geom_vline(aes(xintercept=quantile_05), linetype="dashed")
  + labs(x="", y="", title="Distribution of daily Apple stock returns")
  + scale_x_continuous(labels=percent_format())
)
apple_returns_figure.show()

In [ ]:
pd.DataFrame(returns["ret"].describe()).round(3).T

In [ ]:
(returns["ret"]
  .groupby(returns["date"].dt.year)
  .describe()
  .round(3)
)

In [ ]:
symbols = tf.download_data(
  domain="constituents", 
  index="Dow Jones Industrial Average"
)

In [ ]:
prices_daily = tf.download_data(
  domain="stock_prices", 
  symbols=symbols["symbol"].tolist(),
  start_date="2023-01-01", 
  end_date="2023-12-31"
)

In [ ]:
from mizani.breaks import date_breaks
from mizani.formatters import date_format

prices_figure = (
  ggplot(prices_daily, aes(y="adjusted_close", x="date", color="symbol"))
  + geom_line()
  + scale_x_datetime(date_breaks="5 years", date_labels="%Y")
  + labs(x="", y="", color="", title="Stock prices of DOW index constituents")
  + theme(legend_position="none")
)
prices_figure.show()

In [ ]:
returns_daily = (prices_daily
  .assign(ret=lambda x: x.groupby("symbol")["adjusted_close"].pct_change())
  .get(["symbol", "date", "ret"])
  .dropna(subset="ret")
)

(returns_daily
  .groupby("symbol")["ret"]
  .describe()
  .round(3)
)

In [ ]:
returns_monthly = (returns_daily
  .assign(date=returns_daily["date"].dt.to_period("M").dt.to_timestamp())
  .groupby(["symbol", "date"], as_index=False)
  .agg(ret=("ret", lambda x: np.prod(1 + x) - 1))
)

In [ ]:
apple_daily = (returns_daily
  .query("symbol == 'AAPL'")
  .assign(frequency="Daily")
)

apple_monthly = (returns_monthly
  .query("symbol == 'AAPL'")
  .assign(frequency="Monthly")
)

apple_returns = pd.concat([apple_daily, apple_monthly], ignore_index=True)

apple_returns_figure = (
  ggplot(apple_returns, aes(x="ret", fill="frequency"))
  + geom_histogram(position="identity", bins=50)
  + labs(
      x="", y="", fill="Frequency",
      title="Distribution of Apple returns across different frequencies"
  )
  + scale_x_continuous(labels=percent_format())
  + facet_wrap("frequency", scales="free")
  + theme(legend_position="none")
)
apple_returns_figure.show()

In [ ]:
trading_volume = (prices_daily
  .assign(trading_volume=lambda x: (x["volume"]*x["adjusted_close"])/1e9)
  .groupby("date")["trading_volume"]
  .sum()
  .reset_index()
  .assign(trading_volume_lag=lambda x: x["trading_volume"].shift(periods=1))
)

trading_volume_figure = (
  ggplot(trading_volume, aes(x="date", y="trading_volume"))
  + geom_line()
  + scale_x_datetime(date_breaks="5 years", date_labels="%Y")
  + labs(
      x="", y="",
      title="Aggregate daily trading volume of DOW index constituents in billion USD"
    )
)
trading_volume_figure.show()

In [ ]:
persistence_figure = (
  ggplot(trading_volume, aes(x="trading_volume_lag", y="trading_volume"))
  + geom_point()
  + geom_abline(aes(intercept=0, slope=1), linetype="dashed")
  + labs(
      x="Previous day aggregate trading volume",
      y="Aggregate trading volume",
      title="Persistence in daily trading volume of DOW constituents in billion USD"
    )
)
persistence_figure.show()